In [ ]:
from pathlib import Path
import os, json, random, time, tempfile, traceback, gc, importlib.util, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODULE_PATH = Path("MODE_utilities.py")
if not MODULE_PATH.is_file():
    raise FileNotFoundError(f"Place {MODULE_PATH.name} beside this notebook: {MODULE_PATH.resolve()}")
spec = importlib.util.spec_from_file_location("MODE_utilities_mutation_sensitivity", MODULE_PATH.resolve())
AS = importlib.util.module_from_spec(spec)
spec.loader.exec_module(AS)
assert "val_series_length=series_length - split_idx" in inspect.getsource(AS.train_model_from_barycenters), ("Per-barycenter split is missing from the utilities module")
print("MODE utilities:", MODULE_PATH.resolve())
print("Device:", AS.device)


In [ ]:
MUTATION_PROBS = [0.3, 0.5, 0.7, 0.9, 1.0]
SEEDS = [42, 43, 44, 45, 46]
POPULATION = 20
OFFSPRING = 10
GENERATIONS = 10
SBX_PROB = 0.9
SBX_ETA = 15
PM_ETA = 20
ASF_WEIGHTS = np.array([0.6, 0.2, 0.2], dtype=float)
HV_REF = np.array([1.1, 1.1, 1.1], dtype=float)

SERIES_LENGTH = 500
N_SERIES = 60
SPLIT = 0.80
GAMMA = 1.0
NORMALIZE = True
MAX_ITER = 50
BARYCENTER_MAX_ITER = 50
MIN_CLUSTER_SIZE = 2
USE_DIVERGENCE = False
D_STAR_TP = 2.6

# Reproduce the manuscript's full MODE bounds. Set on Problem each run.
LOWER_BOUNDS = np.array([5, 1e-4, 8, 2, 50, 1, 2], dtype=float)
UPPER_BOUNDS = np.array([20, 1e-1, 128, 10, 150, 5, 10], dtype=float)

RASPBERRY_FOLDER = Path("Rasbery_Pi")
OUTPUT_DIR = Path("Experimental results") / "MODE_RPi_CPU_mutation_sensitivity"
TRACK_ENERGY = False          # Optional: True if CodeCarbon is installed and desired
FORCE_RERUN = False
FAIL_FAST = False
SAVE_PREDICTIONS = False      # Keep false to avoid 25x bulky predictions
MAKE_PLOTS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS_CSV = OUTPUT_DIR / "mutation_runs.csv"
SERIES_CSV = OUTPUT_DIR / "mutation_per_series_rmse.csv"
FAILURES_CSV = OUTPUT_DIR / "mutation_failures.csv"
CONFIG_JSON = OUTPUT_DIR / "experiment_config.json"

assert len(MUTATION_PROBS) == 5 and len(SEEDS) == 5
assert len(ASF_WEIGHTS) == 3
assert np.all(LOWER_BOUNDS < UPPER_BOUNDS)
print("Planned MODE runs:", len(MUTATION_PROBS) * len(SEEDS))
print("Results folder:", OUTPUT_DIR.resolve())


In [ ]:
all_data = AS.read_raspberry_pi_dataset(
    series_length=SERIES_LENGTH,
    base_folder=str(RASPBERRY_FOLDER),
    column="cpu",
)
all_data = np.asarray(all_data[:N_SERIES], dtype=np.float64)
if all_data.shape != (N_SERIES, SERIES_LENGTH):
    raise ValueError(f"Expected {N_SERIES} CPU profiles of length {SERIES_LENGTH}; got {all_data.shape}")
if not np.isfinite(all_data).all():
    raise ValueError("Nonfinite data in selected RPi profiles")
split_idx = int(SPLIT * SERIES_LENGTH)
offline_data = all_data[:, :split_idx]
online_data = all_data[:, split_idx:]
print("All:", all_data.shape, "Development:", offline_data.shape, "Held-out test:", online_data.shape)


In [ ]:
def atomic_csv(frame, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_path = tempfile.mkstemp(dir=target.parent, suffix=".csv", prefix="._tmp_")
    os.close(fd)
    try:
        frame.to_csv(temp_path, index=False)
        os.replace(temp_path, target)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

def read_csv(path):
    return pd.read_csv(path) if Path(path).is_file() else pd.DataFrame()

def replace_run(old, new, run_id):
    if old.empty:
        return new.copy()
    if "run_id" not in old:
        raise ValueError("Existing results lack 'run_id'; use a fresh OUTPUT_DIR")
    return pd.concat([old.loc[old.run_id.astype(str) != str(run_id)], new], ignore_index=True)

def set_seeds(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    AS.torch.manual_seed(int(seed))
    if AS.torch.cuda.is_available():
        AS.torch.cuda.manual_seed_all(int(seed))
        AS.torch.backends.cudnn.benchmark = False
        AS.torch.backends.cudnn.deterministic = True

CONFIG = {
    "mutation_probs": MUTATION_PROBS, "seeds": SEEDS,
    "population": POPULATION, "offspring": OFFSPRING, "generations": GENERATIONS,
    "sbx_prob": SBX_PROB, "sbx_eta": SBX_ETA, "pm_eta": PM_ETA,
    "asf_weights": ASF_WEIGHTS.tolist(), "hv_ref": HV_REF.tolist(),
    "series_length": SERIES_LENGTH, "n_series": N_SERIES,
    "outer_split": SPLIT, "inner_split": SPLIT, "gamma": GAMMA,
    "normalize": NORMALIZE, "max_iter": MAX_ITER,
    "barycenter_max_iter": BARYCENTER_MAX_ITER,
    "min_cluster_size": MIN_CLUSTER_SIZE, "use_divergence": USE_DIVERGENCE,
    "cp_threshold": D_STAR_TP, "lower_bounds": LOWER_BOUNDS.tolist(),
    "upper_bounds": UPPER_BOUNDS.tolist(),
    "split_method": "chronological_per_barycenter",
    "data_shape": list(all_data.shape),
}
if CONFIG_JSON.exists():
    previous = json.loads(CONFIG_JSON.read_text(encoding="utf8"))
    if previous != CONFIG:
        raise ValueError("Configuration changed in an existing output directory. Use a NEW OUTPUT_DIR or restore original settings.")
else:
    CONFIG_JSON.write_text(json.dumps(CONFIG, indent=2), encoding="utf8")
print("Configuration verified:", CONFIG_JSON)


In [ ]:
def run_mode_with_mutation_probability(development_data, mutation_probability, seed):
    set_seeds(seed)
    problem = AS.Clustering_centralized(
        offline_data=development_data,
        gamma=GAMMA,
        max_iter=MAX_ITER,
        barycenter_max_iter=BARYCENTER_MAX_ITER,
        normalize=NORMALIZE,
        use_divergence=USE_DIVERGENCE,
        min_cluster_size=MIN_CLUSTER_SIZE,
        split=SPLIT,
        D_star_tp=D_STAR_TP,
    )
    # Enforce manuscript bounds for all five mutation conditions.
    problem.xl = LOWER_BOUNDS.copy()
    problem.xu = UPPER_BOUNDS.copy()
    assert problem.n_var == 7 and problem.n_obj == 3

    algorithm = AS.NSGA2(
        pop_size=POPULATION,
        n_offsprings=OFFSPRING,
        sampling=AS.LHS(),
        crossover=AS.SBX(prob=SBX_PROB, eta=SBX_ETA),
        mutation=AS.PM(prob=float(mutation_probability), eta=PM_ETA),
        eliminate_duplicates=True,
    )
    termination = AS.get_termination("n_gen", GENERATIONS)
    AS._sync_device()
    started = time.perf_counter()
    result = AS.pymoo_minimize(
        problem, algorithm, termination,
        seed=int(seed), save_history=True, verbose=False,
    )
    AS._sync_device()
    elapsed = time.perf_counter() - started
    problem.timing["total_mode_sec"] = elapsed
    components = sum(problem.timing[k] for k in (
        "clustering_sec", "training_sec", "validation_sec", "stability_sec"
    ))
    problem.timing["measured_components_sec"] = components
    problem.timing["other_sec"] = max(0.0, elapsed - components)
    if result.F is None or len(result.F) == 0:
        raise RuntimeError("MODE produced no feasible Pareto front")
    return result, problem, elapsed


def history_objective_arrays(result):
    """Save every population objective matrix for shared-bounds HV recomputation."""
    out = {}
    for g, state in enumerate(result.history):
        values = state.pop.get("F")
        out[f"generation_{g:03d}"] = np.asarray(values, dtype=np.float64).reshape(-1, 3)
    return out


def execute_one(mutation_probability, seed):
    run_id = f"pm{mutation_probability:.1f}_seed{int(seed)}"
    (res, problem, duration), energy = AS.run_with_energy_tracking(
        lambda: run_mode_with_mutation_probability(
            offline_data, mutation_probability, seed
        ),
        label=f"MODE_sensitivity_{run_id}", enable=TRACK_ENERGY,
    )
    instance, model, config, info = AS.ApplyingASF(
        res, problem, weights=ASF_WEIGHTS, return_full=True
    )
    per_series = []
    for i, ts in enumerate(online_data):
        metrics = AS.evaluate_one_individual_series(
            ML=instance, series_raw=ts,
            sdtw_normalize=NORMALIZE, return_raw_scale=True,
        )
        per_series.append({
            "run_id": run_id, "p_m": float(mutation_probability),
            "seed": int(seed), "series_id": i,
            "rmse_raw": float(metrics["rmse_raw"]),
            "rmse_normalized": float(metrics["rmse_used"]),
            "n_test_predictions": len(metrics["targets_raw"]),
        })
    per_series = pd.DataFrame(per_series)
    if len(per_series) != N_SERIES or not np.isfinite(per_series.rmse_raw).all():
        raise ValueError("Missing or invalid per-series raw-scale RMSE results")
    t = problem.timing
    run = {
        "run_id": run_id, "dataset_id": "RPi_CPU",
        "p_m": float(mutation_probability),
        "p_var": 1 / 7,
        "effective_gene_selection_probability": float(mutation_probability) / 7,
        "seed": int(seed),
        "n_series": N_SERIES,
        "mean_raw_rmse": float(per_series.rmse_raw.mean()),
        "std_across_profiles_raw_rmse": float(per_series.rmse_raw.std(ddof=1)),
        "median_raw_rmse": float(per_series.rmse_raw.median()),
        "total_mode_sec": float(t["total_mode_sec"]),
        "clustering_sec": float(t["clustering_sec"]),
        "training_sec": float(t["training_sec"]),
        "validation_sec": float(t["validation_sec"]),
        "stability_sec": float(t["stability_sec"]),
        "other_sec": float(t["other_sec"]),
        "n_clusterings": int(t["n_clusterings"]),
        "n_trainings": int(t["n_trainings"]),
        "n_validations": int(t["n_validations"]),
        "n_stability_evals": int(t["n_stability_evals"]),
        "selected_k": int(config["num_clusters"]),
        "data_size_points": int(config["num_clusters"] * offline_data.shape[1]),
        "lstm_training_points": int(config["num_clusters"] * int(SPLIT * offline_data.shape[1])),
        "selected_f1_validation_rmse": float(info["best_F"][0]),
        "selected_f2_k": float(info["best_F"][1]),
        "selected_f3_stability": float(info["best_F"][2]),
        "final_pareto_front_size": int(len(np.asarray(res.F).reshape(-1, 3))),
        "energy_wh": float(energy.get("energy_wh", np.nan)),
        "emissions_kg": float(energy.get("emissions_kg", np.nan)),
        "tracker_available": bool(energy.get("tracker_available", False)),
    }
    for k in ("window", "learning_rate", "batch_size", "hidden_dim", "epochs", "layer_dim"):
        run[f"selected_{k}"] = config[k]
    objective_history = history_objective_arrays(res)
    return pd.DataFrame([run]), per_series, objective_history


In [ ]:
plan = [(float(pm), int(seed)) for pm in MUTATION_PROBS for seed in SEEDS]
for run_idx, (pm, seed) in enumerate(plan, 1):
    run_id = f"pm{pm:.1f}_seed{seed}"
    runs_existing = read_csv(RUNS_CSV)
    series_existing = read_csv(SERIES_CSV)
    n_detail = int((series_existing.run_id == run_id).sum()) if not series_existing.empty else 0
    history_file = OUTPUT_DIR / f"{run_id}_history.npz"
    done = (not runs_existing.empty and run_id in runs_existing.run_id.astype(str).values
            and n_detail == N_SERIES and history_file.exists())
    if done and not FORCE_RERUN:
        print(f"[{run_idx:02d}/{len(plan)}] SKIP {run_id}", flush=True)
        continue

    print(f"[{run_idx:02d}/{len(plan)}] START {run_id}", flush=True)
    began = time.perf_counter()
    try:
        run_row, profile_rows, history = execute_one(pm, seed)
        # Save immutable history before committing the run marker.
        tmp_history = OUTPUT_DIR / f".{run_id}_history_tmp.npz"
        np.savez_compressed(tmp_history, **history)
        os.replace(tmp_history, history_file)
        atomic_csv(replace_run(read_csv(SERIES_CSV), profile_rows, run_id), SERIES_CSV)
        atomic_csv(replace_run(read_csv(RUNS_CSV), run_row, run_id), RUNS_CSV)
        print(f"   SAVED: time={run_row.total_mode_sec.iloc[0]:.1f}s, "
              f"RMSE={run_row.mean_raw_rmse.iloc[0]:.4f}, "
              f"K={run_row.selected_k.iloc[0]}", flush=True)
    except Exception as exc:
        error = pd.DataFrame([{
            "run_id": run_id, "p_m": pm, "seed": seed,
            "exception": repr(exc),
            "elapsed_sec": time.perf_counter() - began,
            "traceback": traceback.format_exc(),
        }])
        atomic_csv(pd.concat([read_csv(FAILURES_CSV), error], ignore_index=True), FAILURES_CSV)
        print(f"   FAILED {run_id}: {exc}", flush=True)
        if FAIL_FAST:
            raise
    finally:
        gc.collect()
        if AS.torch.cuda.is_available():
            AS.torch.cuda.empty_cache()
print("Finished processing sensitivity plan.")


In [ ]:
from pymoo.indicators.hv import HV
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

REQUIRE_ALL_RUNS_FOR_HV = True

def shared_hypervolume():
    runs = read_csv(RUNS_CSV)
    expected = [f"pm{pm:.1f}_seed{seed}" for pm, seed in plan]
    completed = set(runs.run_id.astype(str)) if not runs.empty else set()
    missing = sorted(set(expected) - completed)
    if missing and REQUIRE_ALL_RUNS_FOR_HV:
        raise RuntimeError(f"Missing {len(missing)} runs. Run all conditions before shared-bounds HV. Examples: {missing[:5]}")
    available = [rid for rid in expected if rid in completed and (OUTPUT_DIR / f"{rid}_history.npz").exists()]
    if not available:
        raise RuntimeError("No completed run histories available")

    pooled = []
    for rid in available:
        with np.load(OUTPUT_DIR / f"{rid}_history.npz") as hist:
            pooled.extend(np.asarray(hist[k], dtype=float).reshape(-1, 3) for k in sorted(hist.files))
    full = np.vstack(pooled)
    full = full[np.isfinite(full).all(axis=1)]
    ideal = np.min(full, axis=0)
    nadir = np.max(full, axis=0)
    denominator = np.where(np.abs(nadir - ideal) < 1e-12, 1.0, nadir - ideal)
    indicator = HV(ref_point=HV_REF)
    records = []
    for rid in available:
        row = runs.loc[runs.run_id.astype(str) == rid].iloc[-1]
        with np.load(OUTPUT_DIR / f"{rid}_history.npz") as hist:
            for generation_index, key in enumerate(sorted(hist.files), 1):
                pop = np.asarray(hist[key], dtype=float).reshape(-1, 3)
                pop = pop[np.isfinite(pop).all(axis=1)]
                if len(pop):
                    normalized = np.clip((pop - ideal) / denominator, 0, 1)
                    nd_indices = NonDominatedSorting().do(normalized, only_non_dominated_front=True)
                    nd = normalized[nd_indices]
                    valid = nd[np.all(nd < HV_REF, axis=1)]
                    value = float(indicator(valid)) if len(valid) else 0.0
                else:
                    value = np.nan
                records.append({"run_id": rid, "p_m": row.p_m, "seed": int(row.seed),
                                "generation": generation_index, "hv_shared": value})
    hist_frame = pd.DataFrame(records)
    final_frame = hist_frame.sort_values("generation").groupby("run_id", as_index=False).tail(1)
    updated = runs.drop(columns=["hv_final_shared"], errors="ignore").merge(
        final_frame[["run_id", "hv_shared"]].rename(columns={"hv_shared": "hv_final_shared"}),
        on="run_id", how="left",
    )
    atomic_csv(hist_frame, OUTPUT_DIR / "mutation_hv_by_generation.csv")
    atomic_csv(updated, RUNS_CSV)
    (OUTPUT_DIR / "hv_shared_bounds.json").write_text(json.dumps({
        "ideal": ideal.tolist(), "nadir": nadir.tolist(), "reference": HV_REF.tolist(),
        "completed_run_ids": available,
        "note": "shared min-max over all generations and all completed runs",
    }, indent=2), encoding="utf8")
    print("HV ideal:", ideal, "HV nadir:", nadir, "reference:", HV_REF)
    print("Saved shared HV for", len(available), "runs")
    return hist_frame, updated

hv_history, runs_with_hv = shared_hypervolume()


In [ ]:
runs = read_csv(RUNS_CSV)
if runs.empty:
    raise RuntimeError("No runs available")
metrics = [
    "mean_raw_rmse", "hv_final_shared", "total_mode_sec", "selected_k",
    "clustering_sec", "training_sec", "validation_sec", "stability_sec", "other_sec",
]
aggregations = {metric: ["mean", "std", "min", "max"] for metric in metrics if metric in runs.columns}
grouped = runs.groupby("p_m", sort=True).agg(aggregations)
grouped.columns = [f"{metric}_{stat}" for metric, stat in grouped.columns]
grouped.insert(0, "n_seeds", runs.groupby("p_m").size())
grouped = grouped.reset_index()
atomic_csv(grouped, OUTPUT_DIR / "mutation_summary_by_probability.csv")
print(grouped[[x for x in (
    "p_m", "n_seeds", "mean_raw_rmse_mean", "mean_raw_rmse_std",
    "hv_final_shared_mean", "hv_final_shared_std",
    "total_mode_sec_mean", "total_mode_sec_std", "selected_k_mean"
) if x in grouped.columns]].to_string(index=False))
print("Summary CSV:", (OUTPUT_DIR / "mutation_summary_by_probability.csv").resolve())


In [ ]:
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

def plot_by_probability(metric, ylabel, filename):
    runs = read_csv(RUNS_CSV)
    if metric not in runs.columns:
        print(f"Missing {metric}: skipping {filename}")
        return
    data = [runs.loc[runs.p_m == pm, metric].dropna().to_numpy(dtype=float) for pm in MUTATION_PROBS]
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.boxplot(data, tick_labels=[f"{pm:g}" for pm in MUTATION_PROBS], showmeans=True)
    for idx, values in enumerate(data, 1):
        ax.scatter(np.repeat(idx, len(values)), values, s=18, zorder=3, alpha=0.75)
    ax.set_xlabel("Offspring-level mutation probability ($p_m$)")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.show()

if MAKE_PLOTS:
    plot_by_probability("mean_raw_rmse", "Mean test RMSE per run (raw CPU scale)", "mutation_test_rmse")
    plot_by_probability("hv_final_shared", "Final HV (shared normalization)", "mutation_final_hv")
    plot_by_probability("total_mode_sec", "Total MODE execution time (s)", "mutation_execution_time")

    hv_file = OUTPUT_DIR / "mutation_hv_by_generation.csv"
    if hv_file.exists():
        hv = pd.read_csv(hv_file)
        fig, ax = plt.subplots(figsize=(7.2, 4.2))
        for pm in MUTATION_PROBS:
            by_generation = hv.loc[hv.p_m == pm].groupby("generation").hv_shared.agg(["mean", "std"])
            if by_generation.empty:
                continue
            x = by_generation.index.to_numpy(dtype=float)
            mean = by_generation["mean"].to_numpy(dtype=float)
            std = by_generation["std"].fillna(0).to_numpy(dtype=float)
            line, = ax.plot(x, mean, marker="o", markersize=3, label=f"$p_m$={pm:g}")
            ax.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.12)
        ax.set_xlabel("Generation")
        ax.set_ylabel("HV (shared normalization)")
        ax.legend(frameon=False, ncol=2)
        ax.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "mutation_hv_convergence.png", dpi=300, bbox_inches="tight")
        fig.savefig(FIG_DIR / "mutation_hv_convergence.pdf", bbox_inches="tight")
        plt.show()
print("Figures:", FIG_DIR.resolve())
